# Treinamento e avaliação do modelo YOLO

Neste notebook, demonstraremos como treinar um modelo de detecção de objetos YOLO (You Only Look Once) e avaliar seu desempenho.

## 1. Configuração Inicial

Primeiro, importamos as bibliotecas necessárias.

In [ ]:
# Importa a classe YOLO da biblioteca Ultralytics.
from ultralytics import YOLO
import torch

E, então, inicializamos um modelo pré-treinado.

Em nosso caso, utilizamos a versão Yolov11 Nano. Este arquivo e outras versões estão disponíveis para download na documentação do [Yolo11](https://docs.ultralytics.com/pt/models/yolo11/#performance-metrics).

In [ ]:
# 'yolov11n.pt' refere-se à versão 'nano' do modelo YOLOv11, que é leve e rápida.
# Este é o ponto de partida para o fine-tuning.
model = YOLO('yolov11n.pt')

## 2. Treinamento do Modelo

Agora, vamos treinar o modelo no nosso conjunto de dados personalizado. O método `train` é usado com parâmetros específicos para a nossa tarefa. O arquivo `.yaml` contém as informações sobre onde o modelo deve encontrar as imagens e os rótulos de treinamento.

**Observação:** O treinamento pode levar um tempo considerável.

In [ ]:
# Determina automaticamente o dispositivo de processamento (CPU, GPU NVIDIA ou Apple Silicon).
# Isso otimiza a performance ao usar hardware acelerador disponível.
if torch.backends.mps.is_available():
    device = 'mps'  # Para Mac com Apple Silicon (chip M1/M2/M3)
elif torch.cuda.is_available():
    device = 'cuda'  # Para sistemas com GPU NVIDIA (geralmente Windows/Linux)
else:
    device = 'cpu'  # Retorna para CPU se nenhuma GPU for detectada

print(f"Usando dispositivo: {device}")

# Treina o modelo YOLO com seu conjunto de dados personalizado.
# 'data="data.yaml"' aponta para o arquivo de configuração do seu dataset,
# que define os caminhos para as imagens de treino/validação e as classes.
# 'epochs=100' define o número de vezes que o modelo verá todo o conjunto de dados.
# 'imgsz=640' define o tamanho das imagens para treinamento (640x640 pixels).
results = model.train(data="YOLODataset/dataset.yaml", epochs=100, imgsz=640, device=device)

## 2.5 (Opcional) Retomada do treinamento

Se o treinamento foi interrompido, podemos retomar a partir do último ponto de verificação.

In [ ]:
model = YOLO("last.pt")
results = model.train(resume=True)

## 3. Exportação do modelo

Após o treinamento, podemos exportar o modelo para o formato ONNX para implantação.

In [ ]:
model.export(format='onnx')

## 4. Avaliação do modelo

Agora vamos avaliar o desempenho do modelo no conjunto de validação.

In [ ]:
# Valida o modelo treinado.
# Isso avalia o desempenho do modelo em um conjunto de dados de validação,
# gerando métricas como precisão, recall e mAP.
metrics = model.val()

## 5. Inspeção e Exibição de Métricas

Vamos analisar algumas métricas chave da nossa avaliação para entender o desempenho do modelo.

In [ ]:
# Imprime as métricas de desempenho do modelo.
print("Precisão:", metrics.results_dict['metrics/precision(B)'])
print("Recall:", metrics.results_dict['metrics/recall(B)'])
print("mAP50:", metrics.results_dict['metrics/mAP50(B)'])
print("mAP50-95:", metrics.results_dict['metrics/mAP50-95(B)'])

Essas métricas nos dão uma visão geral do desempenho do modelo:

- Precisão: A exatidão das previsões positivas.
- Recall: A fração de positivos reais que foram identificados.
- mAP50: Precisão média com 50% de IoU.
- mAP50-95: Precisão média em diferentes limites de IoU.